In [2]:
import glob
import os
import json
import pandas as pd
import glob
import os
import json
import pandas as pd
import glob
import os
import json
import networkx as nx
import numpy as np
import time
import glob
import os
import json
import scipy as sp
from pathlib import Path


from sidion_qubo import load_data, get_G_ising, solve_ising, export_results_to_json, Ising


def ising2qubo(ising):
    Q, nodes = nx.attr_matrix(ising, edge_attr='J')
    Q = np.triu(Q, 1)
    node_h = nx.get_node_attributes(ising, 'h')
    L = np.array([node_h[node] for node in nodes])
    
    QUBO = 4*Q
    
    offset = -L.sum() + Q.sum()
    np.fill_diagonal(QUBO, 2*L - 2*Q.sum(axis=0) - 2*Q.sum(axis=1))

    return QUBO, offset



In [ ]:
solver_type = 'Advantage2_system1.8'
case_path = '../data/{}/send_{}/m{}/case{}/beta{}/'
#list_of_cases = sorted(glob.glob(case_path.format(solver_type, '8', '12', '1', 0.0)))
list_of_cases = sorted(glob.glob(case_path.format(solver_type, '*', '*', '*', 0.0)))

list_of_cases

In [ ]:
def to_mqlib(Q, file_qplib):
    Q = -(qubo+qubo.T)/2
    QSP = sp.sparse.coo_matrix(Q)
    QUT = sp.sparse.triu(QSP, 1)
    QD = sp.sparse.diags(QSP.diagonal()).tocoo()
    
    n, _ = Q.shape
    m_D = QD.nnz
    m_UT = QUT.nnz

    with open(file_qplib, 'w') as f:
        print(n, m_D+m_UT, file=f)
        for i, j, v in zip(QD.row+1, QD.col+1, QD.data):
            print(i, j, v, file=f)
        for i, j, v in zip(QUT.row+1, QUT.col+1, QUT.data):
            print(i, j, v, file=f)


for ising_type in isings:
    for case in list_of_cases:
        file = f'{case}{ising_type}.json'
        file_QA = file.replace('data', 'results').replace('J_log.json', 'QA_J_log_num_reads_100.json').replace('send_8', 'QA/send_8')
        file_qplib = file.replace('send_8', 'mqlib/send_8').replace('json', 'txt')
        file_qplib_meta = file_qplib.replace('J_log.txt', 'metadata.json')

        print('Reading:', file)
        
        with open(file) as f:
            data = json.load(f)

        with open(file_QA) as f:
            data_QA = json.load(f)

        QA_value = data_QA['value']

        ising = Ising().deserialize(data)
        qubo, offset = ising2qubo(ising)

        print('Writing:', file_qplib)
        print('Writing:', file_qplib_meta)

        Path(os.path.dirname(file_qplib)).mkdir(parents=True, exist_ok=True)
    
        to_mqlib(qubo, file_qplib)

        with open(file_qplib_meta, 'w') as f:
            json.dump({'offset': offset, 'QA_value': QA_value}, f)
        
        print(' ... Done.')

In [3]:
solver_type = 'Advantage2_system1.8'
case_path = '/home/roman/temp/datasets-internal/sidon/data/{}/mqlib/send_{}/m{}/case{}/beta{}/*.txt'
#list_of_cases = sorted(glob.glob(case_path.format(solver_type, '8', '12', '1', 0.0)))
list_of_cases = sorted(glob.glob(case_path.format(solver_type, '*', '*', '*', 0.0)))

list_of_cases

['/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case0/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case1/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case2/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case3/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case4/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case5/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case6/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case7/beta0.0/J_log.txt',
 '/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case8/beta0.0/J_log.txt',
 

In [4]:
script_filename = 'mqlib_hh.sh'
qplib_path = '/home/roman/MQLib/MQLib'

with open(script_filename, 'w') as f:
    print(f'cd {qplib_path}', file=f)
    for file in list_of_cases:
        outfile = file.replace('sidon/data', 'sidon/results') + '.output'
        Path(os.path.dirname(outfile)).mkdir(parents=True, exist_ok=True)
        cmd = f'./bin/MQLib -fQ {file} -hh -r 10 -ps > {outfile}'
        print(file)
        print(f'echo Computing: "{cmd}"', file=f)
        print(cmd, file=f)
        print(f'echo ... Done.', file=f)
    

/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case0/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case1/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case2/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case3/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case4/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case5/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case6/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case7/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/sidon/data/Advantage2_system1.8/mqlib/send_8/m1/case8/beta0.0/J_log.txt
/home/roman/temp/datasets-internal/si